In [ ]:
import pandas as pd
import numpy as np

from snp_analysis_tools_sherlock import *
from coalescence_analysis_tools import *
import iqplot
import bokeh.plotting
import bokeh.io
import holoviews as hv
from holoviews import dim, opts
import bokeh.models
from bokeh.layouts import gridplot

hv.extension('bokeh')

In [ ]:
full_dfp7 = pd.read_csv('selection_coefficients_fitting2.csv').set_index('species-mesocosm')
full_dfp7 = pd.read_csv('selection_coefficients_fitting2_v2.csv').set_index('species-mesocosm')

full_dfp7 = full_dfp7.loc[~full_dfp7['parent_subjects'].isin(['AA-AC/PP',
                                                                    'AC/PP-AE','AC/PP-AF']),:]


In [ ]:
e003_metadata = pd.read_csv('e003_metadata_cultures_round2.csv').drop(columns='Unnamed: 0')
full_dfp7['type_meso']

In [ ]:
cmap_sp = {'s__Flavonifractor plautii': '#77AADD',
 's__Parabacteroides distasonis': '#EE8866',
 's__Bacteroides uniformis': '#FFAABB',
 's__Escherichia coli_D': '#EEDD88', 
 's__Bacteroides thetaiotaomicron': '#99DDFF',
 's__Dorea formicigenerans': '#44BB99',
 's__Parasutterella excrementihominis': '#BBCC33',
 's__Bacteroides_B dorei': '#AAAA00',
 'other': '#DDDDDD'}
full_dfp7['sp_plot'] = full_dfp7['species'].astype(str)
full_dfp7.loc[~full_dfp7['sp_plot'].isin(cmap_sp.keys()),'sp_plot'] = 'other'

In [ ]:
full_dfp7['inoculumn']

In [ ]:
full_dfp7['p7_s']

In [ ]:
from scipy import stats
full_dfp7['species-inoculumn'] = full_dfp7['species_id'].astype(str) + '-' + full_dfp7['inoculumn']
sel_pvals_all = []
freq_pvals_all =[]
sp_ino_all = []
t_stat = []
for sp_inoculumn in full_dfp7['species-inoculumn'].unique():
    df_ino = full_dfp7.loc[full_dfp7['species-inoculumn']== sp_inoculumn,:]
    if len(df_ino['type_meso'].unique())<2:
        continue 
    mGAM_vals= df_ino.loc[df_ino['media'] == 'mGAM','p7_s'].values
    mBHI_vals = df_ino.loc[df_ino['media'] == 'mBHI','p7_s'].values
    t_statistic, p_value = stats.ttest_ind(mGAM_vals[~np.isnan(mGAM_vals)], mBHI_vals[~np.isnan(mBHI_vals)])
    if np.isnan(p_value):
        print(mGAM_vals, mBHI_vals, sp_inoculumn)
    t_stat.append(t_statistic)
    sel_pvals_all.append(p_value)
    sp_ino_all.append(sp_inoculumn)
    mGAM_vals= df_ino.loc[df_ino['media'] == 'mGAM','strain_freq'].values
    mBHI_vals = df_ino.loc[df_ino['media'] == 'mBHI','strain_freq'].values
    t_statistic, p_value = stats.ttest_ind(mGAM_vals[~np.isnan(mGAM_vals)], mBHI_vals[~np.isnan(mBHI_vals)])
    freq_pvals_all.append(p_value)
    
    
df = pd.DataFrame(data={'sel_pval': sel_pvals_all,'freq_pval': freq_pvals_all,
                        'species-inoculumn': sp_ino_all})
df.sort_values(by='sel_pval')
qvals = np.sort(np.array(sel_pvals_all ))
qvals = stats.false_discovery_control(np.sort(qvals[~np.isnan(qvals)]))
dfgood = df.loc[~df['sel_pval'].isna(),:]
dfgood = dfgood.loc[~dfgood['freq_pval'].isna(),:]
dfgood = dfgood.sort_values(by='sel_pval')
dfgood['qvals'] = qvals
dfgood.reset_index()

In [ ]:
len(dfgood.loc[dfgood['qvals']<.05,:])

In [ ]:
17/71

In [ ]:
full_dfp7['count'] = 1
full_dfp7['species-type_meso'] = full_dfp7['species_id'].astype(str) + '-' + full_dfp7['type_meso']
full_dfp7_sum = full_dfp7.groupby(['species-type_meso','type_meso']).sum(numeric_only=True).reset_index()
full_dfp7_sum_good = full_dfp7_sum.loc[full_dfp7_sum['count'] >=1,:]
good_type_mesos = full_dfp7_sum_good['species-type_meso'].values
full_dfp7['species-inoculumn'] = full_dfp7['species_id'].astype(str) + '-' + full_dfp7['inoculumn'].astype(str) 

#full_dfp7 = full_dfp7.set_index('species-type_meso')
test_sets_all = []

for medias in [('mBHI','mGAM')]: 
    media_train = medias[0]
    media_test = medias[1]
    test_set = full_dfp7.loc[full_dfp7['media'] == media_test,:]
    train_set = full_dfp7.loc[full_dfp7['media'] == media_train,:]
    
    test_set = test_set.loc[test_set['species-type_meso'].isin(good_type_mesos),:]
    train_set = train_set.loc[train_set['species-type_meso'].isin(good_type_mesos),:]
    
    good_inoculumns = np.intersect1d(test_set['species-inoculumn'].unique(), train_set['species-inoculumn'].unique())
    test_set = test_set.loc[test_set['species-inoculumn'].isin(good_inoculumns),:]
    train_set = train_set.loc[train_set['species-inoculumn'].isin(good_inoculumns),:]
    
    train_set_med = train_set.groupby(['species-inoculumn', 'inoculumn','parent_media','sp_plot']).median(numeric_only=True)
    train_set_min = train_set.groupby(['species-inoculumn', 'inoculumn','parent_media','sp_plot']).min(numeric_only=True)
    train_set_max = train_set.groupby(['species-inoculumn', 'inoculumn','parent_media','sp_plot']).max(numeric_only=True)
    test_set_min= test_set.groupby(['species-inoculumn', 'inoculumn','parent_media','sp_plot']).min(numeric_only=True)
    test_set_max = test_set.groupby(['species-inoculumn', 'inoculumn','parent_media','sp_plot']).max(numeric_only=True)
    test_set = test_set.groupby(['species-inoculumn', 'inoculumn','parent_media','sp_plot']).median(numeric_only=True)
    
    
    test_set.loc[train_set_med.index.values, 'p7_s_pred'] = train_set_med.loc[train_set_med.index.values, 'p7_s'] 
    test_set.loc[train_set_med.index.values, 'p7_s_pred_min'] = train_set_min.loc[train_set_min.index.values, 'p7_s'] 
    test_set.loc[train_set_med.index.values, 'p7_s_pred_max'] = train_set_max.loc[train_set_max.index.values, 'p7_s'] 

    test_set.loc[test_set_min.index.values, 'p7_s_min'] = test_set_min.loc[test_set_min.index.values, 'p7_s'] 
    test_set.loc[test_set_max.index.values, 'p7_s_max'] = test_set_max.loc[test_set_max.index.values, 'p7_s']
  #  test_set.loc[train_set_med.index.values, 'p7_freq_pred'] = train_set_med.loc[train_set_med.index.values, 'p7_freq'] 
   # test_set.loc[train_set_med.index.values, 'p0_freq_pred'] = train_set_med.loc[train_set_med.index.values, 'p0_freq'] 
   # test_set.loc[train_set_med.index.values, 'p3_s_pred'] = train_set_med.loc[train_set_med.index.values, 'p3_s'] 
   # test_set.loc[train_set_med.index.values, 'p7_s_fr5_pred'] = train_set_med.loc[train_set_med.index.values, 'p7_s_fr5'] 

    
    test_set['media'] = media_test
    test_set['media_train'] = media_train
    test_sets_all.append(test_set)

test_sets_all = pd.concat(test_sets_all).reset_index()
test_sets_all['p7_s_pred_gen'] = test_sets_all['p7_s_pred']/np.log2(200)
test_sets_all['p7_s_gen'] = test_sets_all['p7_s']/np.log2(200)


#test_sets_all['env'] = test_sets_all['parent_media'] + '-' +  test_sets_all['media']
#full_dfp7_nonzero_V2 = full_dfp7_nonzero.loc[full_dfp7_nonzero['parent_subjects'].isin(['AC/PP-AE','AA-AE','AA-AF']),:]
#test_sets_all_good = test_sets_all.loc[test_sets_all['mesocosm'] == 'A9-AA-AC/PP-mGAM-mBHI',:]
#test_sets_all_good = test_sets_all_good.loc[test_sets_all_good['species_id'] == 101346,:]
test_sets_all = test_sets_all.sort_values(by='parent_media',ascending=False)
test_sets_all['same_dir'] = (test_sets_all['p7_s_pred_gen']>0)== (test_sets_all['p7_s_gen']>0)
dfgood['sig'] = dfgood['qvals']<.05
test_sets_all = test_sets_all.loc[test_sets_all['species-inoculumn'].isin(dfgood['species-inoculumn']),:]
test_sets_all = pd.concat([test_sets_all.set_index('species-inoculumn'), dfgood.set_index('species-inoculumn')],axis=1)
test_sets_all['is_sig'] = 'p > .05'
test_sets_all.loc[test_sets_all['sig'],'is_sig'] = 'p < .05'

scatter1 = hv.Scatter(test_sets_all, kdims =  'p7_s_pred', vdims=['p7_s','qvals']).opts(size=10, 
                                                                                    color = 'qvals',
                                                                                       width = 600,
                                                                                       height = 500,
                                                                                                #  cmap=cmap_sp,
                                                                                           # cmap = bokeh.palettes.Colorblind[6][::-1],
                                                                                     # cmap = bokeh.palettes.Set3[11][::-1],
                                                                                                        colorbar=True, 
                                                                                                #   logx=True, logy=True,
                                                                                            legend_position='right',
                                                                                         title = 'p7_s pred from reps colored by freq',
                                                                                                  alpha = .75,
                                                                                                  show_legend=True,
                                                                                               xlim=(-2.5, 2.5), ylim=(-2.5,2.5))
                                                                                             #xlim=(-0.05,1.05),ylim=(-0.05,1.05))
                                                                                                #    xlim=(-1.5, 1.5), ylim=(-1.5, 1.5))

# (x0, y0, x1, y1)
seg = hv.Segments(test_sets_all, ['p7_s_pred_min', 
                         'p7_s', 
                         'p7_s_pred_max', 'p7_s', ]).opts(color='black')

seg2 = hv.Segments(test_sets_all, ['p7_s_pred', 
                         'p7_s_min', 
                         'p7_s_pred', 'p7_s_max', ]).opts(color='black')
scatter1  = scatter1*seg*seg2
scatter = hv.render(scatter1)
scatter.line(np.arange(-200,200)/10, np.arange(-200,200)/10,color = 'black', )
scatter.legend.visible = True
scatter.title.text = 'Selection passages 0 to 7'
scatter.xaxis.axis_label = 'mBHI'
scatter.yaxis.axis_label = 'mGAM'

low_box = bokeh.models.PolyAnnotation(fill_alpha=0.2, fill_color='grey',
                                     xs=[0,0,-3,-3],
                                     ys = [0,3,3,0])

high_box = bokeh.models.PolyAnnotation(fill_alpha=0.2, fill_color='grey',
                                     xs=[0,0,3,3],
                                     ys = [0,-3,-3,0])


scatter.add_layout(low_box)
scatter.add_layout(high_box)
#scatter.output_backend = "svg"
#scatter.legend.location = 'right'
bokeh.io.show(scatter)

In [ ]:
from scipy import stats
full_dfp7['species-inoculumn'] = full_dfp7['species_id'].astype(str) + '-' + full_dfp7['inoculumn']
sel_pvals_all = []
freq_pvals_all =[]
sp_ino_all = []
t_stat = []
for sp_inoculumn in full_dfp7['species-inoculumn'].unique():
    df_ino = full_dfp7.loc[full_dfp7['species-inoculumn']== sp_inoculumn,:]
    if len(df_ino['type_meso'].unique())<2:
        continue 
    mGAM_vals= df_ino.loc[df_ino['media'] == 'mGAM','p3_s'].values
    mBHI_vals = df_ino.loc[df_ino['media'] == 'mBHI','p3_s'].values
    t_statistic, p_value = stats.ttest_ind(mGAM_vals[~np.isnan(mGAM_vals)], mBHI_vals[~np.isnan(mBHI_vals)])
    if np.isnan(p_value):
        print(mGAM_vals, mBHI_vals, sp_inoculumn)
    t_stat.append(t_statistic)
    sel_pvals_all.append(p_value)
    sp_ino_all.append(sp_inoculumn)
    mGAM_vals= df_ino.loc[df_ino['media'] == 'mGAM','p3_freq'].values
    mBHI_vals = df_ino.loc[df_ino['media'] == 'mBHI','p3_freq'].values
    t_statistic, p_value = stats.ttest_ind(mGAM_vals[~np.isnan(mGAM_vals)], mBHI_vals[~np.isnan(mBHI_vals)])
    freq_pvals_all.append(p_value)
    
    
df = pd.DataFrame(data={'sel_pval': sel_pvals_all,'freq_pval': freq_pvals_all,
                        'species-inoculumn': sp_ino_all})
df.sort_values(by='sel_pval')
qvals = np.sort(np.array(sel_pvals_all ))
qvals = stats.false_discovery_control(np.sort(qvals[~np.isnan(qvals)]))
dfgood = df.loc[~df['sel_pval'].isna(),:]
dfgood = dfgood.loc[~dfgood['freq_pval'].isna(),:]
dfgood = dfgood.sort_values(by='sel_pval')
dfgood['qvals'] = qvals
dfgood.reset_index()


In [ ]:
full_dfp7['count'] = 1
full_dfp7['species-type_meso'] = full_dfp7['species_id'].astype(str) + '-' + full_dfp7['type_meso']
full_dfp7_sum = full_dfp7.groupby(['species-type_meso','type_meso']).sum(numeric_only=True).reset_index()
full_dfp7_sum_good = full_dfp7_sum.loc[full_dfp7_sum['count'] >=1,:]
good_type_mesos = full_dfp7_sum_good['species-type_meso'].values
full_dfp7['species-inoculumn'] = full_dfp7['species_id'].astype(str) + '-' + full_dfp7['inoculumn'].astype(str) 

#full_dfp7 = full_dfp7.set_index('species-type_meso')
test_sets_all = []

for medias in [('mBHI','mGAM')]: 
    media_train = medias[0]
    media_test = medias[1]
    test_set = full_dfp7.loc[full_dfp7['media'] == media_test,:]
    train_set = full_dfp7.loc[full_dfp7['media'] == media_train,:]
    
    test_set = test_set.loc[test_set['species-type_meso'].isin(good_type_mesos),:]
    train_set = train_set.loc[train_set['species-type_meso'].isin(good_type_mesos),:]
    
    good_inoculumns = np.intersect1d(test_set['species-inoculumn'].unique(), train_set['species-inoculumn'].unique())
    test_set = test_set.loc[test_set['species-inoculumn'].isin(good_inoculumns),:]
    train_set = train_set.loc[train_set['species-inoculumn'].isin(good_inoculumns),:]
    
    train_set_med = train_set.groupby(['species-inoculumn', 'inoculumn','parent_media','sp_plot']).median(numeric_only=True)
    train_set_min = train_set.groupby(['species-inoculumn', 'inoculumn','parent_media','sp_plot']).min(numeric_only=True)
    train_set_max = train_set.groupby(['species-inoculumn', 'inoculumn','parent_media','sp_plot']).max(numeric_only=True)
    test_set_min= test_set.groupby(['species-inoculumn', 'inoculumn','parent_media','sp_plot']).min(numeric_only=True)
    test_set_max = test_set.groupby(['species-inoculumn', 'inoculumn','parent_media','sp_plot']).max(numeric_only=True)
    test_set = test_set.groupby(['species-inoculumn', 'inoculumn','parent_media','sp_plot']).median(numeric_only=True)
    
    
    test_set.loc[train_set_med.index.values, 'p3_s_pred'] = train_set_med.loc[train_set_med.index.values, 'p3_s'] 
    test_set.loc[train_set_med.index.values, 'p3_s_pred_min'] = train_set_min.loc[train_set_min.index.values, 'p3_s'] 
    test_set.loc[train_set_med.index.values, 'p3_s_pred_max'] = train_set_max.loc[train_set_max.index.values, 'p3_s'] 

    test_set.loc[test_set_min.index.values, 'p3_s_min'] = test_set_min.loc[test_set_min.index.values, 'p3_s'] 
    test_set.loc[test_set_max.index.values, 'p3_s_max'] = test_set_max.loc[test_set_max.index.values, 'p3_s']
  #  test_set.loc[train_set_med.index.values, 'p7_freq_pred'] = train_set_med.loc[train_set_med.index.values, 'p7_freq'] 
   # test_set.loc[train_set_med.index.values, 'p0_freq_pred'] = train_set_med.loc[train_set_med.index.values, 'p0_freq'] 
   # test_set.loc[train_set_med.index.values, 'p3_s_pred'] = train_set_med.loc[train_set_med.index.values, 'p3_s'] 
   # test_set.loc[train_set_med.index.values, 'p7_s_fr5_pred'] = train_set_med.loc[train_set_med.index.values, 'p7_s_fr5'] 

    
    test_set['media'] = media_test
    test_set['media_train'] = media_train
    test_sets_all.append(test_set)

test_sets_all = pd.concat(test_sets_all).reset_index()


#test_sets_all['env'] = test_sets_all['parent_media'] + '-' +  test_sets_all['media']
#full_dfp7_nonzero_V2 = full_dfp7_nonzero.loc[full_dfp7_nonzero['parent_subjects'].isin(['AC/PP-AE','AA-AE','AA-AF']),:]
#test_sets_all_good = test_sets_all.loc[test_sets_all['mesocosm'] == 'A9-AA-AC/PP-mGAM-mBHI',:]
#test_sets_all_good = test_sets_all_good.loc[test_sets_all_good['species_id'] == 101346,:]
test_sets_all = test_sets_all.sort_values(by='parent_media',ascending=False)
#test_sets_all['same_dir'] = (test_sets_all['p3_s_pred_gen']>0)== (test_sets_all['p7_s_gen']>0)
dfgood['sig'] = dfgood['qvals']<.05
test_sets_all = test_sets_all.loc[test_sets_all['species-inoculumn'].isin(dfgood['species-inoculumn']),:]
test_sets_all = pd.concat([test_sets_all.set_index('species-inoculumn'), dfgood.set_index('species-inoculumn')],axis=1)
test_sets_all['is_sig'] = 'p > .05'
test_sets_all.loc[test_sets_all['sig'],'is_sig'] = 'p < .05'

scatter1 = hv.Scatter(test_sets_all, kdims =  'p3_s_pred', vdims=['p3_s','qvals']).opts(size=10, 
                                                                                color = 'qvals',
                                                                                       width = 600,
                                                                                       height = 500,
                                                                                         
                                                                                                #  cmap=cmap_sp,
                                                                                          #  cmap = bokeh.palettes.Colorblind[6][::-1],
                                                                                     # cmap = bokeh.palettes.Set3[11][::-1],
                                                                                                        colorbar=True, 
                                                                                                #   logx=True, logy=True,
                                                                                            legend_position='right',
                                                                                         title = 'p7_s pred from reps colored by freq',
                                                                                                  alpha = .75,
                                                                                                  show_legend=True,
                                                                                               xlim=(-2.5, 2.5), ylim=(-2.5,2.5))
                                                                                             #xlim=(-0.05,1.05),ylim=(-0.05,1.05))
                                                                                                #    xlim=(-1.5, 1.5), ylim=(-1.5, 1.5))

# (x0, y0, x1, y1)
seg = hv.Segments(test_sets_all, ['p3_s_pred_min', 
                         'p3_s', 
                         'p3_s_pred_max', 'p3_s', ]).opts(color='black')

seg2 = hv.Segments(test_sets_all, ['p3_s_pred', 
                         'p3_s_min', 
                         'p3_s_pred', 'p3_s_max', ]).opts(color='black')
scatter1  = scatter1*seg*seg2
scatter = hv.render(scatter1)
scatter.line(np.arange(-200,200)/10, np.arange(-200,200)/10,color = 'black', )
scatter.legend.visible = True
scatter.title.text = 'Selection passages 0 to 3'
scatter.xaxis.axis_label = 'mBHI'
scatter.yaxis.axis_label = 'mGAM'

low_box = bokeh.models.PolyAnnotation(fill_alpha=0.2, fill_color='grey',
                                     xs=[0,0,-3,-3],
                                     ys = [0,3,3,0])

high_box = bokeh.models.PolyAnnotation(fill_alpha=0.2, fill_color='grey',
                                     xs=[0,0,3,3],
                                     ys = [0,-3,-3,0])


scatter.add_layout(low_box)
scatter.add_layout(high_box)
#scatter.output_backend = "svg"
#scatter.legend.location = 'right'
bokeh.io.show(scatter)

In [ ]:
#test_sets_all[['p7_s','p7_s_pred', 'same_dir', 'sel_pval']].sort_values(by='sel_pval')

In [ ]:
full_dfp7['count'] = 1
full_dfp7['species-type_meso'] = full_dfp7['species_id'].astype(str) + '-' + full_dfp7['type_meso']
full_dfp7_sum = full_dfp7.groupby(['species-type_meso','type_meso']).sum(numeric_only=True).reset_index()
full_dfp7_sum_good = full_dfp7_sum.loc[full_dfp7_sum['count'] >=1,:]
good_type_mesos = full_dfp7_sum_good['species-type_meso'].values
full_dfp7['species-inoculumn'] = full_dfp7['species_id'].astype(str) + '-' + full_dfp7['inoculumn'].astype(str) 

#full_dfp7 = full_dfp7.set_index('species-type_meso')
test_sets_all = []
for medias in [('mBHI','mGAM')]: 
    media_train = medias[0]
    media_test = medias[1]
    test_set = full_dfp7.loc[full_dfp7['media'] == media_test,:]
    train_set = full_dfp7.loc[full_dfp7['media'] == media_train,:]
    
    test_set = test_set.loc[test_set['species-type_meso'].isin(good_type_mesos),:]
    train_set = train_set.loc[train_set['species-type_meso'].isin(good_type_mesos),:]
    
    good_inoculumns = np.intersect1d(test_set['species-inoculumn'].unique(), train_set['species-inoculumn'].unique())
    test_set = test_set.loc[test_set['species-inoculumn'].isin(good_inoculumns),:]
    train_set = train_set.loc[train_set['species-inoculumn'].isin(good_inoculumns),:]
    
    train_set_med = train_set.groupby(['species-inoculumn', 'inoculumn','parent_media','sp_plot']).median(numeric_only=True)
    train_set_min = train_set.groupby(['species-inoculumn', 'inoculumn','parent_media','sp_plot']).min(numeric_only=True)
    train_set_max = train_set.groupby(['species-inoculumn', 'inoculumn','parent_media','sp_plot']).max(numeric_only=True)
    test_set_min= test_set.groupby(['species-inoculumn', 'inoculumn','parent_media','sp_plot']).min(numeric_only=True)
    test_set_max = test_set.groupby(['species-inoculumn', 'inoculumn','parent_media','sp_plot']).max(numeric_only=True)
    test_set = test_set.groupby(['species-inoculumn', 'inoculumn','parent_media','sp_plot']).median(numeric_only=True)
    
    
    test_set.loc[train_set_med.index.values, 'p3_s_pred'] = train_set_med.loc[train_set_med.index.values, 'p3_s'] 
    test_set.loc[train_set_med.index.values, 'p3_s_pred_min'] = train_set_min.loc[train_set_min.index.values, 'p3_s'] 
    test_set.loc[train_set_med.index.values, 'p3_s_pred_max'] = train_set_max.loc[train_set_max.index.values, 'p3_s'] 

    test_set.loc[test_set_min.index.values, 'p3_s_min'] = test_set_min.loc[test_set_min.index.values, 'p3_s'] 
    test_set.loc[test_set_max.index.values, 'p3_s_max'] = test_set_max.loc[test_set_max.index.values, 'p3_s']
  #  test_set.loc[train_set_med.index.values, 'p7_freq_pred'] = train_set_med.loc[train_set_med.index.values, 'p7_freq'] 
   # test_set.loc[train_set_med.index.values, 'p0_freq_pred'] = train_set_med.loc[train_set_med.index.values, 'p0_freq'] 
   # test_set.loc[train_set_med.index.values, 'p3_s_pred'] = train_set_med.loc[train_set_med.index.values, 'p3_s'] 
   # test_set.loc[train_set_med.index.values, 'p7_s_fr5_pred'] = train_set_med.loc[train_set_med.index.values, 'p7_s_fr5'] 

    
    test_set['media'] = media_test
    test_set['media_train'] = media_train
    test_sets_all.append(test_set)

test_sets_all = pd.concat(test_sets_all).reset_index()
#test_sets_all['p7_s_pred_gen'] = test_sets_all['p7_s_pred']/np.log2(200)
#test_sets_all['p7_s_gen'] = test_sets_all['p7_s']/np.log2(200)


#test_sets_all['env'] = test_sets_all['parent_media'] + '-' +  test_sets_all['media']
#full_dfp7_nonzero_V2 = full_dfp7_nonzero.loc[full_dfp7_nonzero['parent_subjects'].isin(['AC/PP-AE','AA-AE','AA-AF']),:]
#test_sets_all_good = test_sets_all.loc[test_sets_all['mesocosm'] == 'A9-AA-AC/PP-mGAM-mBHI',:]
#test_sets_all_good = test_sets_all_good.loc[test_sets_all_good['species_id'] == 101346,:]
test_sets_all = test_sets_all.sort_values(by='parent_media',ascending=False)
test_sets_all['same_dir'] = (test_sets_all['p3_s_pred']>0)== (test_sets_all['p3_s']>0)
scatter1 = hv.Scatter(test_sets_all, kdims =  'p3_s_pred', vdims=['p3_s','same_dir']).opts(size=10, 
                                                                                # color = 'media',
                                                                                       width = 600,
                                                                                       height = 500,
                                                                                                #  cmap=cmap_sp,
                                                                                            cmap = bokeh.palettes.Colorblind[6][::-1],
                                                                                     # cmap = bokeh.palettes.Set3[11][::-1],
                                                                                                        colorbar=True, 
                                                                                                #   logx=True, logy=True,
                                                                                            legend_position='right',
                                                                                         title = 'p7_s pred from reps colored by freq',
                                                                                                  alpha = .75,
                                                                                                  show_legend=False,
                                                                                               xlim=(-2.5, 2.5), ylim=(-2.5,2.5))
                                                                                             #xlim=(-0.05,1.05),ylim=(-0.05,1.05))
                                                                                                #    xlim=(-1.5, 1.5), ylim=(-1.5, 1.5))

# (x0, y0, x1, y1)
seg = hv.Segments(test_sets_all, ['p3_s_pred_min', 
                         'p3_s', 
                         'p3_s_pred_max', 'p3_s', ]).opts(color='black')

seg2 = hv.Segments(test_sets_all, ['p3_s_pred', 
                         'p3_s_min', 
                         'p3_s_pred', 'p3_s_max', ]).opts(color='black')
scatter1  = scatter1*seg*seg2
scatter = hv.render(scatter1)
scatter.line(np.arange(-200,200)/10, np.arange(-200,200)/10,color = 'black', )
scatter.legend.visible = True
scatter.title.text = 'Selection passages 0 to 3'
scatter.xaxis.axis_label = 'mBHI'
scatter.yaxis.axis_label = 'mGAM'

low_box = bokeh.models.PolyAnnotation(fill_alpha=0.2, fill_color='grey',
                                     xs=[0,0,-3,-3],
                                     ys = [0,3,3,0])

high_box = bokeh.models.PolyAnnotation(fill_alpha=0.2, fill_color='grey',
                                     xs=[0,0,3,3],
                                     ys = [0,-3,-3,0])


scatter.add_layout(low_box)
scatter.add_layout(high_box)
#scatter.output_backend = "svg"
#scatter.legend.location = 'right'
bokeh.io.show(scatter)

In [ ]:
test_sets_all['same_dir'] = (test_sets_all['p3_s_pred']>0)== (test_sets_all['p3_s']>0)

In [ ]:
1-test_sets_all['same_dir'].sum()/len(test_sets_all['same_dir'])

In [ ]:
1-test_sets_all['same_dir'].sum()/40

In [ ]:
4/44